# 🏥 Sympriority — Voice-to-Triage Pipeline (Ministral-3B Edition)

**Completely standalone notebook — run all cells top-to-bottom on Google Colab (free T4 GPU).**

| Stage | Model | VRAM |
|---|---|---|
| 🎤 Speech  | `openai/whisper-medium` | ~2 GB |
| 🤖 Symptoms  | `mistralai/Ministral-3-3B-Instruct-2512` | ~6 GB |
| **Total** | — | **~9 GB** (T4 has 15 GB) |

**Features:**
- Speaks any language (Hindi, Tamil, Telugu, English) → auto-translated to English by Whisper
- 4-level clinical triage: Critical / High / Moderate / Low
- Color-coded triage card with priority score, recommended action, and reasoning
- Gradio UI with `share=True` link (works from Colab)

## Cell 1 — Install Dependencies
Run this first (takes ~1-2 min on Colab). Safe to re-run — pip is idempotent.

In [ ]:
# ── Install all required packages ─────────────────────────────
# ffmpeg: needed by Whisper to decode audio (mic/file input)
# openai-whisper: speech-to-text model (runs locally on GPU)
# gradio>=4.0: web UI — mic input, text output, HTML triage card
# transformers + accelerate: run Ministral-3B locally on GPU
!apt-get install -y -q ffmpeg
!pip install -q openai-whisper "gradio>=4.0" transformers accelerate

print("✅ All dependencies installed.")

## Cell 2 — Load Models (Whisper + Ministral-3B)
⚠️ **Replace `HF_TOKEN`** with a fresh token from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens) before running.

This cell loads both models into GPU memory (~8 GB total). Takes 2-5 minutes on first run (downloads weights). Subsequent runs in the same Colab session reuse cached weights.

In [ ]:
import torch, whisper, gradio as gr, json, re, os
from transformers import pipeline as hf_pipeline

# ── HF Token — REPLACE with your fresh token from huggingface.co/settings/tokens ──
HF_TOKEN = "your_hf_token_here"   
os.environ["HF_TOKEN"] = HF_TOKEN

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Device: {device.upper()}")

# ── Load Whisper-medium (~2 GB VRAM) ──────────────────────────
# VRAM reuse guard: safe to re-run cell without reloading the model
if "asr_model" not in dir() or asr_model is None:
    print("⏳ Loading whisper-medium ...")
    asr_model = whisper.load_model("medium", device=device)
    print("✅ Whisper-medium loaded")
else:
    print("✅ Whisper-medium already in session — reusing (no extra VRAM)")

# ── Load Ministral-3B-Instruct (~6 GB VRAM) ───────────────────
# mistralai/Ministral-3-3B-Instruct-2512 — Apache 2.0, no license agreement needed.
# device_map="auto" distributes layers across available GPU/CPU automatically.
print("⏳ Loading Ministral-3B-Instruct-2512 (local GPU) ...")
triage_pipe = hf_pipeline(
    "text-generation",
    model="mistralai/Ministral-3-3B-Instruct-2512",
    token=HF_TOKEN,
    device_map="auto",
    torch_dtype=torch.float16
)
print("✅ Ministral-3B loaded — total VRAM in use: ~8 GB")

## Cell 3 — Triage Pipeline Functions
Defines the clinical triage prompt and two functions:
- `run_triage(symptoms)` — calls Ministral-3B, returns parsed JSON  
- `full_pipeline(audio_path)` — Whisper transcription → triage → HTML card

In [ ]:
# ── Clinical triage prompt with strict 4-level criteria ───────
# Ministral (Mistral family) works best with a single user message
# that embeds both the system instructions and the patient input.
TRIAGE_PROMPT = """You are a clinical triage AI for a hospital OPD. Assess patient symptom urgency accurately using the criteria below.

RISK LEVEL GUIDELINES — apply strictly:

CRITICAL (score 9-10): Immediately life-threatening. Emergency intervention required NOW.
  Examples: chest pain with sweating/arm pain, difficulty breathing or choking, stroke signs
  (face drooping, slurred speech, sudden arm weakness), unconsciousness, severe allergic
  reaction (throat swelling), uncontrolled bleeding, poisoning, suspected heart attack.

HIGH (score 6-8): Serious — needs doctor within 30-60 minutes.
  Examples: fever above 39.5°C/103°F, severe abdominal pain, head injury with confusion,
  suspected fracture, persistent vomiting with dehydration, child with high fever, chest
  tightness without other cardiac signs, signs of infection spreading.

MODERATE (score 3-5): Needs medical attention today, not an emergency.
  Examples: fever 38°C-39.5°C, ear/throat infection, urinary tract infection, mild-moderate
  abdominal pain, minor wounds needing stitches, back pain, persistent diarrhea,
  toothache, mild allergic rash.

LOW (score 1-2): Routine care, no urgency.
  Examples: common cold, mild headache without other symptoms, minor cuts or bruises,
  mild cough without fever, runny nose, routine medication refill, skin rash without
  swelling or spreading.

CRITICAL RULE: A simple cold, mild fever, or mild headache alone is ALWAYS Low or Moderate — NEVER Critical or High.
Assign scores honestly based on clinical evidence in the symptoms, not assumptions.

Respond ONLY with valid JSON — no text outside the JSON block. Include ALL 8 fields:
{
  "risk_level": "Critical" | "High" | "Moderate" | "Low",
  "priority_score": <integer 1-10, 10 = most critical>,
  "recommended_action": "<specific immediate action, e.g. 'Call emergency services immediately' or 'Visit OPD today'>",
  "reasoning": "<1-2 sentences explaining the risk level based on specific symptoms>",
  "department": "<hospital department or specialist, e.g. 'Emergency / Cardiology', 'General OPD', 'ENT', 'Orthopaedics'>",
  "vital_signs_to_monitor": "<comma-separated vitals to watch, e.g. 'pulse rate, blood pressure, oxygen saturation, respiratory rate'>",
  "warning_signs": "<2-3 specific symptoms that would immediately escalate urgency and require emergency care>",
  "first_aid": "<immediate practical steps the patient or bystander should take RIGHT NOW before reaching hospital>"
}"""


def run_triage(symptom_text: str) -> dict:
    """Send symptoms to Ministral-3B and return parsed triage JSON."""
    messages = [
        {"role": "user", "content": f"{TRIAGE_PROMPT}\n\nPatient symptoms:\n{symptom_text}"}
    ]
    result = triage_pipe(messages, max_new_tokens=450, do_sample=False, return_full_text=False)
    raw = result[0]["generated_text"]
    if isinstance(raw, list):
        raw = raw[-1]["content"]
    raw = raw.strip()

    match = re.search(r'\{[\s\S]*?\}', raw)
    if match:
        try:
            return json.loads(match.group())
        except json.JSONDecodeError:
            pass
    return {
        "risk_level": "Unknown",
        "priority_score": 0,
        "recommended_action": "Please review manually.",
        "reasoning": raw,
        "department": "—",
        "vital_signs_to_monitor": "—",
        "warning_signs": "—",
        "first_aid": "—"
    }


RISK_COLORS = {
    "Critical": "#c0392b",
    "High":     "#e67e22",
    "Moderate": "#d4ac0d",
    "Low":      "#27ae60",
    "Unknown":  "#7f8c8d"
}

RISK_BG = {
    "Critical": "#fdf2f2",
    "High":     "#fef6ed",
    "Moderate": "#fefdf0",
    "Low":      "#f0faf4",
    "Unknown":  "#f4f6f7"
}


def _section(icon, title, body, border_color="#1a5276"):
    """Render one labelled info section with a left-border accent."""
    return f"""
        <div style="margin-bottom:10px; padding:10px 14px; background:#ffffff;
                    border-radius:6px; border-left: 4px solid {border_color};
                    box-shadow:0 1px 3px rgba(0,0,0,0.06);">
            <div style="color:{border_color}; font-weight:600; font-size:0.85rem;
                        text-transform:uppercase; letter-spacing:0.5px; margin-bottom:5px;">
                {icon}&nbsp; {title}
            </div>
            <div style="color:#1a252f; font-size:0.95rem; line-height:1.55;">{body}</div>
        </div>"""


def full_pipeline(audio_path):
    """End-to-end: audio file -> English transcription + 6-section HTML triage card."""
    if audio_path is None:
        return "⚠️ No audio recorded. Click the mic button first.", ""

    # Stage 1: Whisper — transcribe + translate to English in one pass
    try:
        result = asr_model.transcribe(
            audio_path,
            task="translate",
            beam_size=3,
            fp16=False
        )
        english = result["text"].strip()
    except Exception as e:
        return f"❌ Transcription error: {str(e)}", ""

    if not english:
        return "⚠️ Could not transcribe audio. Speak louder/closer and retry.", ""

    # Stage 2: Ministral-3B triage
    try:
        triage = run_triage(english)
    except Exception as e:
        return english, f"<p style='color:red'>❌ Triage error: {str(e)}</p>"

    risk    = triage.get("risk_level", "Unknown")
    score   = triage.get("priority_score", 0)
    action  = triage.get("recommended_action", "")
    reason  = triage.get("reasoning", "")
    dept    = triage.get("department", "—")
    vitals  = triage.get("vital_signs_to_monitor", "—")
    warning = triage.get("warning_signs", "—")
    aid     = triage.get("first_aid", "—")
    color   = RISK_COLORS.get(risk, "#7f8c8d")
    bg      = RISK_BG.get(risk, "#f4f6f7")

    triage_html = f"""
    <div style="font-family:'Segoe UI',Arial,sans-serif; padding:18px;
                background:{bg}; border-radius:10px;
                border: 2px solid {color}; margin-top:4px;">

        <!-- Header: risk badge + score + department pill -->
        <div style="display:flex; flex-wrap:wrap; align-items:center;
                    gap:12px; margin-bottom:16px;">
            <div style="background:{color}; color:#fff; padding:8px 20px;
                        border-radius:6px; font-size:1.15rem; font-weight:700;
                        letter-spacing:0.5px;">
                {risk.upper()}
            </div>
            <div style="font-size:1rem; color:#1a252f;">
                Priority Score:&nbsp;
                <span style="color:{color}; font-weight:700; font-size:1.2rem;">
                    {score}<span style="font-size:0.82rem; color:#5d6d7e;">/10</span>
                </span>
            </div>
            <div style="margin-left:auto; background:#ffffff; border:1px solid {color};
                        color:{color}; padding:5px 14px; border-radius:20px;
                        font-size:0.85rem; font-weight:600; white-space:nowrap;">
                🏥 {dept}
            </div>
        </div>

        {_section("📋", "Recommended Action", action, color)}
        {_section("🧠", "Clinical Reasoning", reason, "#5d6d7e")}
        {_section("💊", "Immediate First Aid / Steps", aid, "#1a5276")}
        {_section("📊", "Vital Signs to Monitor", vitals, "#117a65")}
        {_section("⚠️", "Warning Signs — Escalate Immediately If", warning, "#c0392b")}

    </div>
    """
    return english, triage_html


print("✅ Pipeline functions defined — 6-section triage card ready.")


## Cell 5 — Gradio UI
Launches the full voice-to-triage interface. After running this cell, Colab will print a public `share.gradio.live` link — click it to open the UI in any browser.

**How to use:**
1. Click the microphone icon and allow microphone access
2. Speak your symptoms (any language)
3. Click Stop, then click **▶ Transcribe & Analyse**
4. Wait ~10 seconds for transcription + triage result

In [ ]:
triage_css = """
body, .gradio-container, .gradio-container > .main,
.gradio-container .prose { background-color: #f0f4f8 !important; }

#mic-box, #mic-box .wrap, #mic-box > div {
    background: #ffffff !important;
    border: 2px solid #1a5276 !important;
    border-radius: 10px !important;
}
#run-btn, #run-btn button {
    background: #c0392b !important; color: #ffffff !important;
    border: none !important; border-radius: 8px !important;
    font-size: 1rem !important; font-weight: 700 !important;
    width: 100% !important; padding: 14px !important;
}
#run-btn button:hover { background: #a93226 !important; }

#eng-out, #eng-out .wrap, #eng-out > div,
#eng-out textarea, #eng-out [data-testid="textbox"] {
    background: #ffffff !important; color: #1a252f !important;
}
#eng-out textarea {
    border: 2px solid #1a5276 !important; border-radius: 8px !important;
    font-size: 1rem !important; line-height: 1.6 !important; padding: 12px !important;
}
#eng-out label, #eng-out span {
    color: #1a5276 !important; font-weight: 600 !important; background: transparent !important;
}
"""

with gr.Blocks(css=triage_css, title="Sympriority — Triage") as demo:

    gr.HTML("""
        <div style="background:linear-gradient(135deg,#c0392b 0%,#1a5276 100%);
                    border-radius:12px; padding:22px 28px; margin-bottom:14px;">
            <h1 style="color:#fff; margin:0 0 6px 0; font-size:1.75rem;
                       font-family:'Segoe UI',Arial,sans-serif; font-weight:700;">
                🏥 Sympriority — Patient Triage
            </h1>
            <p style="color:#dce9f5; margin:0; font-size:0.95rem;
                      font-family:'Segoe UI',Arial,sans-serif;">
                Speak your symptoms in
                <b style="color:#fff;">English, Hindi, Telugu, or Tamil</b>.
                AI assesses risk and recommends action.
            </p>
        </div>
    """)

    audio_in = gr.Audio(
        sources=["microphone"],
        type="filepath",
        label="🎤  Click mic → speak symptoms → click Stop",
        elem_id="mic-box"
    )

    run_btn = gr.Button("▶  Transcribe & Analyse", elem_id="run-btn", size="lg")

    english_out = gr.Textbox(
        label="📝  Symptom Description (English)",
        lines=3,
        placeholder="Transcribed English text will appear here ...",
        elem_id="eng-out"
    )

    triage_out = gr.HTML(
        label="🩺  Triage Assessment",
        value="<p style='color:#7f8c8d; font-family:Segoe UI,Arial,sans-serif; padding:12px;'>"
              "Triage result will appear here after analysis ...</p>"
    )

    gr.HTML(
        "<p style='color:#5d6d7e; font-size:0.82rem; text-align:center; margin-top:6px;'>"
        "Stage 1: openai/whisper-medium &nbsp;·&nbsp; "
        "Stage 2: mistralai/Ministral-3-3B-Instruct-2512 (local GPU)"
        "</p>"
    )

    run_btn.click(
        fn=full_pipeline,
        inputs=[audio_in],
        outputs=[english_out, triage_out]
    )

demo.launch(share=True)